In [ ]:
# ==========================================================
# TASK 4: PORTFOLIO OPTIMIZATION USING MODERN PORTFOLIO THEORY
# TSLA + SPY + BND
# ==========================================================


# -----------------------------
# Import Libraries
# -----------------------------

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import yfinance as yf

from scipy.optimize import minimize

import seaborn as sns





# ==========================================================
# 1. Download Historical Data
# ==========================================================


assets = [
    "TSLA",
    "SPY",
    "BND"
]


data = yf.download(
    assets,
    start="2020-01-01",
    end="2026-01-01"
)


prices = data["Close"]


prices = prices.dropna()



print(prices.head())





# ==========================================================
# 2. Calculate Historical Daily Returns
# ==========================================================


returns = prices.pct_change().dropna()



display(
    returns.head()
)





# ==========================================================
# 3. Expected Returns
# ==========================================================


# -----------------------------
# TSLA Forecast Return
# -----------------------------


current_tsla_price = prices["TSLA"].iloc[-1]


# ==========================================================
# LOAD TSLA FUTURE FORECAST FROM TASK 3
# ==========================================================

future_forecast = pd.read_csv(
    "tsla_future_forecast.csv",
    index_col=0,
    parse_dates=True
)


print("Future forecast loaded")

display(
    future_forecast.head()
)



tsla_expected_return = (

    (
        future_tsla_price -
        current_tsla_price
    )

    /

    current_tsla_price

)





# -----------------------------
# SPY and BND historical returns
# -----------------------------


spy_expected_return = (

    returns["SPY"].mean()

    *

    252

)



bnd_expected_return = (

    returns["BND"].mean()

    *

    252

)





expected_returns = np.array([

    tsla_expected_return,

    spy_expected_return,

    bnd_expected_return

])




print("\nExpected Annual Returns")

print(
    pd.Series(
        expected_returns,
        index=["TSLA","SPY","BND"]
    )
)





# ==========================================================
# 4. Covariance Matrix
# ==========================================================


cov_matrix = returns.cov() * 252



print("\nAnnualized Covariance Matrix")

display(
    cov_matrix
)





# Heatmap

plt.figure(
    figsize=(7,5)
)


sns.heatmap(
    cov_matrix,
    annot=True,
    cmap="coolwarm"
)


plt.title(
    "Annualized Covariance Matrix"
)


plt.show()





# ==========================================================
# 5. Portfolio Functions
# ==========================================================


def portfolio_return(weights):

    return np.dot(
        weights,
        expected_returns
    )



def portfolio_volatility(weights):

    return np.sqrt(
        np.dot(
            weights.T,
            np.dot(
                cov_matrix.values,
                weights
            )
        )
    )



def sharpe_ratio(weights):

    return (

        portfolio_return(weights)

        /

        portfolio_volatility(weights)

    )





# ==========================================================
# 6. Generate Efficient Frontier
# ==========================================================


num_portfolios = 5000


results = []


weights_record = []



for i in range(num_portfolios):


    weights = np.random.random(3)


    weights = weights / np.sum(weights)



    ret = portfolio_return(
        weights
    )


    vol = portfolio_volatility(
        weights
    )


    sharpe = ret / vol



    results.append(
        [
            ret,
            vol,
            sharpe
        ]
    )


    weights_record.append(
        weights
    )



results = pd.DataFrame(
    results,
    columns=[
        "Return",
        "Volatility",
        "Sharpe"
    ]
)





# ==========================================================
# 7. Maximum Sharpe Portfolio
# ==========================================================


max_sharpe = results.loc[
    results["Sharpe"].idxmax()
]


max_weights = weights_record[
    results["Sharpe"].idxmax()
]





# ==========================================================
# 8. Minimum Volatility Portfolio
# ==========================================================


min_vol = results.loc[
    results["Volatility"].idxmin()
]


min_weights = weights_record[
    results["Volatility"].idxmin()
]





# ==========================================================
# 9. Efficient Frontier Plot
# ==========================================================


plt.figure(
    figsize=(10,6)
)


plt.scatter(

    results["Volatility"],

    results["Return"],

    c=results["Sharpe"],

    cmap="viridis",

    s=5

)


plt.colorbar(
    label="Sharpe Ratio"
)



plt.scatter(

    max_sharpe["Volatility"],

    max_sharpe["Return"],

    color="red",

    marker="*",

    s=300,

    label="Maximum Sharpe Portfolio"

)



plt.scatter(

    min_vol["Volatility"],

    min_vol["Return"],

    color="blue",

    marker="X",

    s=200,

    label="Minimum Volatility Portfolio"

)



plt.xlabel(
    "Portfolio Volatility"
)


plt.ylabel(
    "Expected Annual Return"
)


plt.title(
    "Efficient Frontier - TSLA SPY BND"
)


plt.legend()

plt.grid()

plt.show()





# ==========================================================
# 10. Portfolio Results
# ==========================================================


portfolio_summary = pd.DataFrame({

    "Asset":
    [
        "TSLA",
        "SPY",
        "BND"
    ],

    "Maximum Sharpe Weights":
    max_weights,

    "Minimum Volatility Weights":
    min_weights

})



display(
    portfolio_summary
)





print("\nMaximum Sharpe Portfolio")

print(
    "Expected Return:",
    round(max_sharpe["Return"]*100,2),
    "%"
)


print(
    "Volatility:",
    round(max_sharpe["Volatility"]*100,2),
    "%"
)


print(
    "Sharpe Ratio:",
    round(max_sharpe["Sharpe"],3)
)





print("\nMinimum Volatility Portfolio")

print(
    "Expected Return:",
    round(min_vol["Return"]*100,2),
    "%"
)


print(
    "Volatility:",
    round(min_vol["Volatility"]*100,2),
    "%"
)





# ==========================================================
# 11. Recommendation
# ==========================================================


print("""

PORTFOLIO RECOMMENDATION

The recommended portfolio is the Maximum Sharpe Ratio Portfolio because
it provides the highest risk-adjusted return. This portfolio balances the
higher expected return from Tesla's forecast with the diversification
benefits of SPY and the lower volatility characteristics of BND.

""")

[*********************100%***********************]  3 of 3 completed

Ticker            BND         SPY       TSLA
Date                                        
2020-01-02  69.255760  296.125305  28.684000
2020-01-03  69.503189  293.882904  29.534000
2020-01-06  69.420723  295.004089  30.102667
2020-01-07  69.362976  294.174744  31.270666
2020-01-08  69.255760  295.742432  32.809334


Ticker,BND,SPY,TSLA
Date,,,
2020-01-03,0.003573,-0.007572,0.029633
2020-01-06,-0.001187,0.003815,0.019255
2020-01-07,-0.000832,-0.002811,0.038801
2020-01-08,-0.001546,0.005329,0.049205
2020-01-09,0.001191,0.006781,-0.021945


NameError: name 'future_forecast' is not defined